# 🛠️ CATIA V5 Llama 3.2 3B 단일 모델 LLM 질문 대답 성능 평가

## 📌 평가 개요 및 목적
본 노트북은 **Llama 3.2 3B (`unsloth/Llama-3.2-3B-Instruct`) 단일 로컬 모델**을 대상으로 CATIA V5 대표 질문 3문항(`CATIA_RAG_Evaluation_Dataset.csv`)에 대해 **사전학습 지식(Direct LLM)**과 **RAG 적용(RAG On)** 시의 대답 성능을 검증합니다.

### 📊 검증 모델 정보
- **모델명**: Llama 3.2 3B
- **HuggingFace Repo**: `unsloth/Llama-3.2-3B-Instruct` (Meta Llama 3.2 3B 오픈 가중치 LLM)
- **구동 방식**: 100% 로컬 PyTorch GPU/CPU 추론 (API 키 0개)

### 🎯 핵심 검증 포인트
1. **눈으로 문장 확인 (Qualitative)**: 질문별 표준 정답(Ground Truth)과 Llama 3.2 3B의 Direct LLM 답변 및 RAG 답변 문장을 1:1 대조 표로 직접 확인합니다.
2. **정량적 점수 비교 (Quantitative)**: Ground Truth 대비 Direct vs RAG 답변의 **BERTScore (Precision, Recall, F1)** 지표를 계산하여 RAG 적용에 따른 정확도 개선율(+ΔF1)을 점수로 확인합니다.

In [1]:
# 1. 환경 설정 및 주요 모듈 임포트
import os
import sys
import gc
import pandas as pd
import torch
from pathlib import Path
from bert_score import score as bert_score_compute
from IPython.display import HTML, display

# 프로젝트 루트 sys.path 추가
PROJECT_ROOT = Path(".").resolve()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT / ".." / "src").exists():
    PROJECT_ROOT = (PROJECT_ROOT / "..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.rag_chain import RAGPipeline
from src.config import DATA_DIR, CHROMA_DB_DIR

print(f"[System] PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[System] Active GPU: {torch.cuda.get_device_name(0)}")


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[System] PyTorch CUDA Available: True
[System] Active GPU: NVIDIA GeForce RTX 4070 Laptop GPU


In [2]:
# 2. CATIA_RAG_Evaluation_Dataset.csv에서 대표 질문 3개 선별 로딩
csv_file_path = os.path.join(PROJECT_ROOT, "CATIA_RAG_Evaluation_Dataset.csv")

if os.path.exists(csv_file_path):
    df_raw = pd.read_csv(csv_file_path)
    df_raw.rename(columns={
        "ID": "id",
        "카테고리": "category",
        "질문": "question",
        "Qwen 1.5 환각/실패 예상": "expected_failure",
        "RAG 적용 정답 (Ground Truth)": "ground_truth",
        "RAG 문서 근거": "source_doc"
    }, inplace=True)
    
    # 대표 질문 3개 선별 (Q01: Sketcher, Q02: Part Design Rib/Slot, Q10: Specification Tree)
    selected_ids = ["Q01", "Q02", "Q10"]
    df_dataset = df_raw[df_raw["id"].isin(selected_ids)].copy().reset_index(drop=True)
else:
    raise FileNotFoundError(f"Cannot find '{csv_file_path}' dataset file.")

print(f"[Dataset] Selected {len(df_dataset)} Representative CATIA QA Items:")
display(df_dataset[["id", "category", "question", "ground_truth"]])


[Dataset] Selected 3 Representative CATIA QA Items:


,id,category,question,ground_truth
0,Q01,Sketcher Workbench,CATIA V5 Sketcher Workbench의 Operation Tool Ba...,"교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능은 Corner이며, Duplic..."
1,Q02,Part Design (Rib & Slot),CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Pr...,단면 스케치 프로파일과 Center Curve 경로를 따라 3D 솔리드 파이프/리브...
2,Q10,Basic (Tree & Compass Control),CATIA V5 화면의 Specification Tree(스펙 트리)를 화면에서 숨...,Specification Tree를 켜거나 끄는 단축키는 F3 키이며 3D 공간 회...


In [3]:
# 3. Llama 3.2 3B 로컬 모델 단일 답변 생성 (Direct LLM vs RAG LLM)
MODEL_NAME = "Llama 3.2 3B"
REPO_ID = "unsloth/Llama-3.2-3B-Instruct"

print(f"==================================================")
print(f"[Experiment] Initializing Local Model: {MODEL_NAME} ({REPO_ID})")
print(f"==================================================")

direct_answers = []
rag_answers = []
source_citations = []

pipeline = RAGPipeline(model_name=REPO_ID, mode="local")

for idx, row in df_dataset.iterrows():
    qid = row["id"]
    question = row["question"]
    print(f" [{idx+1}/{len(df_dataset)}] Processing {qid}...")
    
    # 1) Direct LLM (RAG Off) 순수 사전학습 지식 답변
    ans_direct = pipeline.answer_direct(question)
    direct_answers.append(ans_direct)
    
    # 2) RAG (RAG On) 매뉴얼 문맥 기반 답변
    res_rag = pipeline.answer_rag(question)
    ans_rag = res_rag["answer"]
    sources = res_rag.get("source_pages", [])
    
    rag_answers.append(ans_rag)
    source_citations.append(", ".join(sources))

print(f"\n[Experiment] {MODEL_NAME} Answer Generation Process Completed!")


[Experiment] Initializing Local Model: Llama 3.2 3B (unsloth/Llama-3.2-3B-Instruct)
[LLMFactory] Initializing LLM 'unsloth/Llama-3.2-3B-Instruct' in mode='local'...
[LLMFactory Local] Downloading/Loading model 'unsloth/Llama-3.2-3B-Instruct' on device 'cuda' (PyTorch Local)...


To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading checkpoint shards: 100%|██████████| 2/2 [00:15<00:00,  7.73s/it]
Device set to use cuda:0


[VectorStore] Initializing Local Embeddings 'sentence-transformers/all-MiniLM-L6-v2'...
[VectorStore] Loading existing Chroma database from: C:\KDT_14\[11]Transformer\project\team-03-project\vect\chroma_db_multimodal


C:\KDT_14\[11]Transformer\project\team-03-project\src\vector_store.py:52: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


 [1/3] Processing Q01...


...

 [2/3] Processing Q02...


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY

 [3/3] Processing Q10...


c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:634: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY311\Lib\site-packages\transformers\generation\configuration_utils.py:629: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.01` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
c:\Users\KDT1\miniconda3\envs\DL_PY


[Experiment] Llama 3.2 3B Answer Generation Process Completed!


## 👁️ 4. 눈으로 직접 보는 생성 문장 비교 셀 (Qualitative Answer Inspection)
3개 질문에 대해 **표준 정답(Ground Truth)**과 **Llama 3.2 3B 모델의 Direct LLM 답변 문장** vs **RAG 적용 답변 문장**을 눈으로 직접 읽고 비교해 볼 수 있는 대조 표입니다.

In [4]:
# 4. Llama 3.2 3B 생성 문장 1:1 Side-by-Side 시각화 표
css_style = """
<style>
    .eval-table {
        width: 100%;
        border-collapse: collapse;
        font-family: 'Segoe UI', Malgun Gothic, sans-serif;
        font-size: 13px;
        margin-top: 10px;
    }
    .eval-table th {
        background-color: #0f172a;
        color: #ffffff;
        text-align: center;
        padding: 10px;
        border: 1px solid #475569;
        font-weight: 600;
    }
    .eval-table td {
        padding: 10px 12px;
        border: 1px solid #cbd5e1;
        vertical-align: top;
        line-height: 1.5;
        word-break: break-word;
    }
    .col-gt { background-color: #f0fdf4; color: #166534; font-weight: 600; }
    .col-direct { background-color: #fef2f2; color: #991b1b; padding: 8px; border-radius: 4px; border: 1px solid #fecdd3; }
    .col-rag { background-color: #eff6ff; color: #1e40af; padding: 8px; border-radius: 4px; border: 1px solid #bfdbfe; font-weight: 500; }
    .badge { display: inline-block; padding: 3px 8px; border-radius: 4px; font-weight: bold; font-size: 11px; margin-bottom: 6px; }
    .badge-direct { background-color: #fda4af; color: #881337; }
    .badge-rag { background-color: #93c5fd; color: #1e3a8a; }
</style>
"""

html_table = css_style + f"""
<div style="overflow-x: auto; border: 1px solid #cbd5e1; border-radius: 6px;">
    <table class="eval-table">
        <thead>
            <tr>
                <th style="width: 60px;">ID</th>
                <th style="width: 220px;">질문 (Question)</th>
                <th style="width: 260px;">표준 정답 (Ground Truth)</th>
                <th style="width: 320px;">{MODEL_NAME} Direct LLM (RAG Off)</th>
                <th style="width: 320px;">{MODEL_NAME} RAG LLM (RAG On)</th>
            </tr>
        </thead>
        <tbody>
"""

for idx, row in df_dataset.iterrows():
    qid = str(row.get("id", ""))
    q = str(row.get("question", ""))
    gt = str(row.get("ground_truth", ""))
    ans_dir = direct_answers[idx]
    ans_rag = rag_answers[idx]
    src = source_citations[idx]
    
    html_table += f"""
            <tr>
                <td style="text-align: center; font-weight: bold;">{qid}</td>
                <td><b>{q}</b></td>
                <td class="col-gt">{gt}</td>
                <td>
                    <span class="badge badge-direct">[RAG Off / Direct LLM]</span><br/>
                    <div class="col-direct">{ans_dir}</div>
                </td>
                <td>
                    <span class="badge badge-rag">[RAG On / 매뉴얼 참조]</span><br/>
                    <div class="col-rag">{ans_rag}</div><br/>
                    <small style="color: #475569;">📍 <b>참조 근거:</b> {src}</small>
                </td>
            </tr>
    """

html_table += """
        </tbody>
    </table>
</div>
"""

display(HTML(html_table))


ID,질문 (Question),표준 정답 (Ground Truth),Llama 3.2 3B Direct LLM (RAG Off),Llama 3.2 3B RAG LLM (RAG On)
Q01,CATIA V5 Sketcher Workbench의 Operation Tool Bar에서 교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능과 선택 요소를 일정 거리만큼 이동 복사하는 기능의 명칭은 무엇인가요?,"교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능은 Corner이며, Duplicate 모드를 정의한 후 선택 요소를 이동 복사하는 기능은 Translate입니다.","[RAG Off / Direct LLM] <|begin_of_text|><|start_header_id|>system<|end_header_id|> Cutting Knowledge Date: December 2023 Today Date: 27 Jul 2026 당신은 친절하고 정확한 CATIA V5 전문 AI 서비스입니다.<|eot_id|><|start_header_id|>user<|end_header_id|> CATIA V5 Sketcher Workbench의 Operation Tool Bar에서 교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능과 선택 요소를 일정 거리만큼 이동 복사하는 기능의 명칭은 무엇인가요?<|eot_id|><|start_header_id|>assistant<|end_header_id|> CATIA V5 Sketcher Workbench의 Operation Tool Bar에서 교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능은 ""Intersect""입니다. Intersect는 두 선분을 교차하는 지점에서 부드럽게 곡선으로 연결하는 기능입니다. 또한, 선택 요소를 일정 거리만큼 이동 복사하는 기능은 ""Move""입니다.","[RAG On / 매뉴얼 참조] <|begin_of_text|><|start_header_id|>system<|end_header_id|> Cutting Knowledge Date: December 2023 Today Date: 27 Jul 2026 당신은 CATIA V5 전문 엔지니어입니다. 아래 매뉴얼 문맥만을 참고하여 답변하세요. 매뉴얼에 정보가 없으면 '제공된 매뉴얼에 관련 내용이 명시되어 있지 않습니다.'라고 답변하세요.<|eot_id|><|start_header_id|>user<|end_header_id|> [매뉴얼 문맥] --- [part design 실습예제.pdf - Page 6] --- 🖼️ [매뉴얼 도면/스크린샷 포함: 4개의 CAD 도면 및 설정 창 캡처 그림이 포함된 페이지입니다.] 6 실습5 [질문] CATIA V5 Sketcher Workbench의 Operation Tool Bar에서 교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능과 선택 요소를 일정 거리만큼 이동 복사하는 기능의 명칭은 무엇인가요?<|eot_id|><|start_header_id|>assistant<|end_header_id|> CATIA V5 Sketcher Workbench의 Operation Tool Bar에서 교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능은 ""스무스 커브""입니다. 또한, 선택 요소를 일정 거리만큼 이동 복사하는 기능은 ""스무스 모션""입니다. 📍 참조 근거: part design 실습예제.pdf (P.6)"
Q02,CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Profile) 단면을 가이드 경로(Center Curve)를 따라 3D 형상으로 생성하는 기능과 그 반대로 홈을 파내는 기능의 명칭은 무엇인가요?,단면 스케치 프로파일과 Center Curve 경로를 따라 3D 솔리드 파이프/리브 형상을 생성하는 기능은 Rib이며 그 반대로 경로를 따라 홈을 파내는 기능은 Slot입니다.,"[RAG Off / Direct LLM] <|begin_of_text|><|start_header_id|>system<|end_header_id|> Cutting Knowledge Date: December 2023 Today Date: 27 Jul 2026 당신은 친절하고 정확한 CATIA V5 전문 AI 서비스입니다.<|eot_id|><|start_header_id|>user<|end_header_id|> CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Profile) 단면을 가이드 경로(Center Curve)를 따라 3D 형상으로 생성하는 기능과 그 반대로 홈을 파내는 기능의 명칭은 무엇인가요?<|eot_id|><|start_header_id|>assistant<|end_header_id|> CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Profile) 단면을 가이드 경로(Center Curve)를 따라 3D 형상으로 생성하는 기능은 ""Extrude""이며, 그 반대로 홈을 파내는 기능은 ""Sweep""입니다.","[RAG On / 매뉴얼 참조] <|begin_of_text|><|start_header_id|>system<|end_header_id|> Cutting Knowledge Date: December 2023 Today Date: 27 Jul 2026 당신은 CATIA V5 전문 엔지니어입니다. 아래 매뉴얼 문맥만을 참고하여 답변하세요. 매뉴얼에 정보가 없으면 '제공된 매뉴얼에 관련 내용이 명시되어 있지 않습니다.'라고 답변하세요.<|eot_id|><|start_header_id|>user<|end_header_id|> [매뉴얼 문맥] --- [part design 실습예제.pdf - Page 6] --- 🖼️ [매뉴얼 도면/스크린샷 포함: 4개의 CAD 도면 및 설정 창 캡처 그림이 포함된 페이지입니다.] 6 실습5 [질문] CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Profile) 단면을 가이드 경로(Center Curve)를 따라 3D 형상으로 생성하는 기능과 그 반대로 홈을 파내는 기능의 명칭은 무엇인가요?<|eot_id|><|start_header_id|>assistant<|end_header_id|> CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Profile) 단면을 가이드 경로(Center Curve)를 따라 3D 형상으로 생성하는 기능은 ""Extrude""이며, 그 반대로 홈을 파내는 기능은 ""Loft""입니다. 📍 참조 근거: part design 실습예제.pdf (P.6)"
Q10,CATIA V5 화면의 Specification Tree(스펙 트리)를 화면에서 숨기거나 다시 켜기 위해 사용되는 키보드 단축키와 3D 나침반의 명칭은 무엇인가요?,Specification Tree를 켜거나 끄는 단축키는 F3 키이며 3D 공간 회전 및 이동에 사용되는 나침반 아이콘의 명칭은 Compass 입니다.,"[RAG Off / Direct LLM] <|begin_of_text|><|start_header_id|>system<|end_header_id|> Cutting Knowledge Date: December 2023 Today Date: 27 Jul 2026 당신은 친절하고 정확한 CATIA V5 전문 AI 서비스입니다.<|eot_id|><|start_header_id|>user<|end_header_id|> CATIA V5 화면의 Specification Tree(스펙 트리)를 화면에서 숨기거나 다시 켜기 위해 사용되는 키보드 단축키와 3D 나침반의 명칭은 무엇인가요?<|eot_id|><|start_header_id|>assistant<|end_header_id|> CATIA V5 화면의 Specification Tree(스펙 트리)를 화면에서 숨기거나 다시 켜기 위해 사용되는 키보드 단축키는 Ctrl + 0, 3D 나침반은 Ctrl + Shift + 0입니다.",[RAG On / 매뉴얼 참조] <|begin_of_text|><|start_header_id|>system<|end_header_id|> Cutti

## 📊 5. 정량적 점수 비교 셀 (BERTScore Metric Evaluation)
Ground Truth(표준 정답) 문장 대비 Llama 3.2 3B의 **Direct LLM 문장** 및 **RAG 문장** 간 **BERTScore Precision, Recall, F1** 점수를 계산하여 RAG 적용 시 수치적 정확도 상승률(+ΔF1)을 측정합니다.

In [5]:
# 5. BERTScore 정량 평가 및 점수 비교 표 계산
ground_truths = df_dataset["ground_truth"].tolist()

print(f"[Metric] Computing BERTScore (Precision, Recall, F1) for {MODEL_NAME}...\n")

# Direct LLM vs Ground Truth BERTScore
P_dir, R_dir, F1_dir = bert_score_compute(cands=direct_answers, refs=ground_truths, lang="ko", verbose=False)
# RAG LLM vs Ground Truth BERTScore
P_rag, R_rag, F1_rag = bert_score_compute(cands=rag_answers, refs=ground_truths, lang="ko", verbose=False)

f1_list_dir = F1_dir.tolist()
f1_list_rag = F1_rag.tolist()

avg_f1_dir = float(F1_dir.mean())
avg_f1_rag = float(F1_rag.mean())
f1_improvement = avg_f1_rag - avg_f1_dir

# 1) 대표 질문별 세부 점수 데이터프레임
df_detail_scores = pd.DataFrame({
    "ID": df_dataset["id"],
    "Category": df_dataset["category"],
    "Direct LLM F1 Score": [round(f, 4) for f in f1_list_dir],
    "RAG LLM F1 Score": [round(f, 4) for f in f1_list_rag],
    "F1 Improvement (+ΔF1)": [round(r - d, 4) for d, r in zip(f1_list_dir, f1_list_rag)]
})

# 2) 평균 점수 요약 표
df_summary_score = pd.DataFrame([{
    "Model": MODEL_NAME,
    "Direct LLM Avg F1": f"{avg_f1_dir:.4f}",
    "RAG LLM Avg F1": f"{avg_f1_rag:.4f}",
    "Total F1 Improvement (+ΔF1)": f"+{f1_improvement:.4f}"
}])

print("======================================================================")
print(f"          {MODEL_NAME} BERTScore QA Performance Summary              ")
print("======================================================================")
display(df_summary_score)
print("\n🔹 [질문별 세부 BERTScore F1 점수 대조]")
display(df_detail_scores)


[Metric] Computing BERTScore (Precision, Recall, F1) for Llama 3.2 3B...

          Llama 3.2 3B BERTScore QA Performance Summary              


,Model,Direct LLM Avg F1,RAG LLM Avg F1,Total F1 Improvement (+ΔF1)
0,Llama 3.2 3B,0.6543,0.6279,+-0.0264



🔹 [질문별 세부 BERTScore F1 점수 대조]


,ID,Category,Direct LLM F1 Score,RAG LLM F1 Score,F1 Improvement (+ΔF1)
0,Q01,Sketcher Workbench,0.6719,0.6480,-0.0239
1,Q02,Part Design (Rib & Slot),0.6595,0.6309,-0.0286
2,Q10,Basic (Tree & Compass Control),0.6314,0.6047,-0.0266


In [6]:
# 6. 전체 문장 및 점수 비교 데이터셋 CSV 저장
df_export = pd.DataFrame({
    "ID": df_dataset["id"],
    "Category": df_dataset["category"],
    "Question": df_dataset["question"],
    "Ground Truth": df_dataset["ground_truth"],
    f"{MODEL_NAME} Direct Answer": direct_answers,
    f"{MODEL_NAME} Direct F1": [round(f, 4) for f in f1_list_dir],
    f"{MODEL_NAME} RAG Answer": rag_answers,
    f"{MODEL_NAME} RAG F1": [round(f, 4) for f in f1_list_rag],
    "Source Citation": source_citations
})

output_csv_path = os.path.join(PROJECT_ROOT, "data", "CATIA_Llama3B_QA_Evaluation_Results.csv")
os.makedirs(os.path.dirname(output_csv_path), exist_ok=True)
df_export.to_csv(output_csv_path, index=False, encoding="utf-8-sig")
print(f"[Saved] Full evaluation results exported to '{output_csv_path}'\n")
display(df_export)


[Saved] Full evaluation results exported to 'C:\KDT_14\[11]Transformer\project\team-03-project\data\CATIA_Llama3B_QA_Evaluation_Results.csv'



,ID,Category,Question,Ground Truth,Llama 3.2 3B Direct Answer,Llama 3.2 3B Direct F1,Llama 3.2 3B RAG Answer,Llama 3.2 3B RAG F1,Source Citation
0,Q01,Sketcher Workbench,CATIA V5 Sketcher Workbench의 Operation Tool Ba...,"교차하는 두 선분을 부드럽게 곡선으로 연결하는 기능은 Corner이며, Duplic...",<|begin_of_text|><|start_header_id|>system<|en...,0.6719,<|begin_of_text|><|start_header_id|>system<|en...,0.6480,part design 실습예제.pdf (P.6)
1,Q02,Part Design (Rib & Slot),CATIA V5 Part Design Workbench에서 2D 닫힌 프로파일(Pr...,단면 스케치 프로파일과 Center Curve 경로를 따라 3D 솔리드 파이프/리브...,<|begin_of_text|><|start_header_id|>system<|en...,0.6595,<|begin_of_text|><|start_header_id|>system<|en...,0.6309,part design 실습예제.pdf (P.6)
2,Q10,Basic (Tree & Compass Control),CATIA V5 화면의 Specification Tree(스펙 트리)를 화면에서 숨...,Specification Tree를 켜거나 끄는 단축키는 F3 키이며 3D 공간 회...,<|begin_of_text|><|start_header_id|>system<|en...,0.6314,<|begin_of_text|><|start_header_id|>system<|en...,0.6047,part design 실습예제.pdf (P.6)
